# HiMoFlow v5.5 — Notebook 02: Evaluation (Generation + V·U·N)

End-to-end generation + V·U·N evaluation using the four checkpoints trained by `01_HiMoFlow_v5_5_training.ipynb`. Wires together:

**Stage 1 (scaffold)** — A1 → A3 → A2  
**Stage 2 (decoration)** — Terminal → assemble_to_smiles → RDKit sanitize

**Goal:** confirm the pipeline produces valid SMILES at non-trivial rate. If it does, we have our first V·U·N number on ZINC250K. If it doesn't, we get a fast feedback loop on which stage breaks the integration.

**Reasonable first-pass V·U·N target:**
- Validity ≥ 50% (lower bound — scaffold-first models have many failure modes)
- Uniqueness ≥ 90% of valid samples
- Novelty ≥ 90% of valid samples not in training

**Anything ≥ these is a real working pipeline.** Below validity 30%, there's a bug somewhere.

**Critical encoding-convention note:**
- A1 outputs `spiro_pos_class` ∈ [0, 8) where **0 = NO_SPIRO** sentinel, classes 1-7 = positions 0-6.
- A3 expects `spiro_pos` ∈ [0, 8) where positions 0-6 = real, **7 = NO_SPIRO** sentinel.
- The v5.5 decoder expects `spiro_atom_positions` with **-1 = NO_SPIRO** sentinel, positions 0-6.

Three different conventions for the same logical field. The smoke does the conversions explicitly.

## 1. Setup — mount Drive, configure paths

In [1]:
!pip install -q rdkit matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 55.8 MB/s eta 0:00:00


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE = '/content/drive/My Drive/machine-learning/generative/MeanFlow/mean-flow-v5.5-ZINC250K'
except ImportError:
    BASE = '.'

import os, sys
print('BASE:', BASE)
assert os.path.isdir(BASE), f'BASE not found: {BASE}'
if BASE not in sys.path:
    sys.path.insert(0, BASE)
os.chdir(BASE)

# ── Checkpoint paths (must match the training notebooks) ─────────
LABELS_PKL = f'{BASE}/dataset_labels_v5_5.pkl'
A1_CKPT  = f'{BASE}/checkpoints_a1_v5_5_3M'
A3_CKPT  = f'{BASE}/checkpoints_a3_v5_5_10M'
A2_CKPT  = f'{BASE}/checkpoints_a2_v5_5_10M'
TERM_CKPT = f'{BASE}/checkpoints_terminal_v5_5_9M'

# Sanity check all checkpoints exist
for name, path in [('A1', A1_CKPT), ('A3', A3_CKPT), ('A2', A2_CKPT), ('Terminal', TERM_CKPT)]:
    best = os.path.join(path, 'best_model.pt')
    if not os.path.isfile(best):
        print(f'✗ {name:8s} best_model.pt MISSING at {best}')
    else:
        size_mb = os.path.getsize(best) / 1e6
        print(f'✓ {name:8s} best_model.pt found ({size_mb:.1f} MB)')

# ── Generation config ─────────────────────────────────────────────
N_SAMPLES        = 1024   # how many molecules to generate
BATCH_SIZE       = 64
CFG_SCALE        = 1.5    # 1.0 = unconditional center, 2.0+ = sharper
TEMPERATURE      = 1.0
A1_STEPS         = 20     # diffusion steps for A1
A2_STEPS         = 20
TERM_STEPS       = 8
SEED             = 0

# Condition sampling: at inference, sample condition vectors from
# the *training distribution* of normalized conditions. We've seen
# this matters — cond=0 (unconditional center) under-generates rings.
# We'll sample N(0,1) which approximates the Z-scored train distribution.
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\ndevice: {device}')
if torch.cuda.is_available():
    print(f'  name: {torch.cuda.get_device_name(0)}')

## 2. Clear stale module imports

In [ ]:
import sys
to_clear = [k for k in list(sys.modules)
            if 'meanflow' in k
            or k.startswith('run_training')]
for k in to_clear:
    del sys.modules[k]
print(f'Cleared {len(to_clear)} stale module(s)')

## 3. Load all four models

Each model loads from its own `best_model.pt`. Total weights ~40M params = ~160 MB fp32, trivial on any modern GPU.

In [ ]:
import torch, json, os
from meanflow.ring_layout_diffusion_v5_5 import build_ring_layout_diffusion_v5_5
from meanflow.branch_topology_diffusion import build_branch_topology_model
from meanflow.ring_atom_diffusion import build_ring_atom_diffusion
from meanflow.terminal_fragment_diffusion import build_fragment_stage2
from run_training_v5_5_a2 import V5_5_NUM_BOND_CLASSES
from run_training_v5_5_terminal import V5_5_NUM_FRAGMENTS, V5_5_NUM_ATOM_TYPES

def _load_state(ckpt_path):
    """Load model state_dict from a checkpoint file.
    Handles both raw state_dict and dict-with-'model'-key formats."""
    sd = torch.load(ckpt_path, map_location=device)
    if isinstance(sd, dict) and 'model' in sd:
        return sd['model'], sd.get('ema', None), sd.get('config', None)
    return sd, None, None

def _apply_ema(model, ema_dict, device):
    """Copy EMA shadow weights into the model. Returns True if applied."""
    if ema_dict is None or 'shadow' not in ema_dict:
        return False
    shadow = ema_dict['shadow']
    for n, p in model.named_parameters():
        if n in shadow:
            p.data.copy_(shadow[n].to(device))
    return True

# ── A1 ───────────────────────────────────────────────────────────
with open(os.path.join(A1_CKPT, 'config.json')) as f: a1_cfg = json.load(f)
a1_state, a1_ema, _ = _load_state(os.path.join(A1_CKPT, 'best_model.pt'))
A1 = build_ring_layout_diffusion_v5_5(
    capacity=a1_cfg['capacity'], condition_dim=2,
).to(device)
A1.load_state_dict(a1_state)
if _apply_ema(A1, a1_ema, device):
    print(f'A1 loaded with EMA weights')
else:
    print(f'A1 loaded (no EMA found in ckpt)')
A1.eval()

# ── A3 ───────────────────────────────────────────────────────────
with open(os.path.join(A3_CKPT, 'config.json')) as f: a3_cfg = json.load(f)
a3_state, a3_ema, _ = _load_state(os.path.join(A3_CKPT, 'best_model.pt'))
A3 = build_branch_topology_model(capacity=a3_cfg['capacity']).to(device)
A3.load_state_dict(a3_state)
if _apply_ema(A3, a3_ema, device):
    print(f'A3 loaded with EMA weights')
else:
    print(f'A3 loaded (no EMA found in ckpt)')
A3.eval()

# ── A2 ───────────────────────────────────────────────────────────
with open(os.path.join(A2_CKPT, 'config.json')) as f: a2_cfg = json.load(f)
a2_state, a2_ema, _ = _load_state(os.path.join(A2_CKPT, 'best_model.pt'))
A2 = build_ring_atom_diffusion(
    capacity=a2_cfg['capacity'], condition_dim=2,
    n_bond_classes=a2_cfg.get('n_bond_classes', V5_5_NUM_BOND_CLASSES),
).to(device)
A2.load_state_dict(a2_state)
if _apply_ema(A2, a2_ema, device):
    print(f'A2 loaded with EMA weights')
else:
    print(f'A2 loaded (no EMA found in ckpt)')
A2.eval()

# ── Terminal ─────────────────────────────────────────────────────
with open(os.path.join(TERM_CKPT, 'config.json')) as f: term_cfg = json.load(f)
term_state, term_ema, _ = _load_state(os.path.join(TERM_CKPT, 'best_model.pt'))
TERM = build_fragment_stage2(
    capacity=term_cfg['capacity'],
    num_fragments=term_cfg.get('num_fragments', V5_5_NUM_FRAGMENTS),
    num_atom_types=term_cfg.get('num_atom_types', V5_5_NUM_ATOM_TYPES),
).to(device)
TERM.load_state_dict(term_state)
if _apply_ema(TERM, term_ema, device):
    print(f'Terminal loaded with EMA weights')
else:
    print(f'Terminal loaded (no EMA found in ckpt)')
TERM.eval()

# Memory check
if torch.cuda.is_available():
    mem_mb = torch.cuda.memory_allocated() / 1e6
    print(f'\nAll models loaded. GPU memory: {mem_mb:.0f} MB')

## 4. Wire the inference pipeline

Defines `generate_batch(condition)` — takes a `(B, 2)` condition tensor and returns a list of SMILES strings (some may be `None` for assembly failures).

In [5]:
import numpy as np
import torch
from meanflow.ring_layout_decoder import (
    decode_v5_5_to_scaffold, aromatic_constraint_mask_v5_5,
    M_MAX, R_MAX, P_MAX, B_LEN_MAX,
)
from meanflow.compose_full_molecule_zinc import (
    assemble_to_smiles, assemble_batch_to_smiles,
)

@torch.no_grad()
def generate_batch(
    condition: torch.Tensor,         # (B, 2) on device
    cfg_scale: float = CFG_SCALE,
    temperature: float = TEMPERATURE,
    a1_steps: int = A1_STEPS,
    a2_steps: int = A2_STEPS,
    term_steps: int = TERM_STEPS,
    seed: int = SEED,
    debug: bool = False,
) -> list:
    """End-to-end generation pipeline. Returns list of SMILES (some may be None).

    Pipeline:
      A1 → spiro convert → A3 → spiro convert → decode_v5_5 → arom_mask
      → A2 → Terminal → assemble_to_smiles → RDKit sanitize
    """
    B = condition.shape[0]

    # ── Stage 1A: A1 (ring layout) ────────────────────────────────
    a1_out = A1.sample(
        condition=condition, n_steps=a1_steps,
        temperature=temperature, cfg_scale=cfg_scale, seed=seed,
    )
    R = a1_out['R']                        # (B, R_MAX)
    F = a1_out['F']                        # (B, R_MAX, R_MAX)
    L = a1_out['L']                        # (B, R_MAX, R_MAX)
    spiro_pos_class = a1_out['spiro_pos_class']  # (B, R_MAX, R_MAX); 0=NO_SPIRO
    if debug:
        print(f'  A1 done: R range [{int(R.min())}, {int(R.max())}], '
              f'F unique {torch.unique(F).cpu().numpy()}')

    # ── Convert A1 spiro_pos_class → A3 spiro_pos encoding ────────
    # A1 convention: 0=NO_SPIRO sentinel, 1..7 = positions 0..6
    # A3 convention: 0..6 = positions, 7=NO_SPIRO sentinel (NO_SPIRO_CLS=7)
    # Convert: (cls == 0) → 7, else (cls - 1)
    a3_spiro_pos = torch.where(
        spiro_pos_class == 0,
        torch.full_like(spiro_pos_class, 7),  # NO_SPIRO_CLS for A3
        spiro_pos_class - 1,
    )

    # ── Stage 1B: A3 (branch topology) ────────────────────────────
    a3_out = A3.sample(
        R=R, F_mat=F, L_mat=L, spiro_pos=a3_spiro_pos,
        condition=condition, cfg_scale=cfg_scale,
        temperature=temperature, post_process=True, seed=seed + 1,
    )
    B_size   = a3_out['B_size']
    B_pos    = a3_out['B_pos']
    B_parent = a3_out['B_parent']
    B_bond   = a3_out['B_bond']
    if debug:
        print(f'  A3 done: B_size range [{int(B_size.min())}, {int(B_size.max())}], '
              f'mean active slots = {(B_size > 0).float().sum(dim=(1,2)).mean():.2f}')

    # ── Convert spiro_pos → spiro_atom_positions (decoder format) ─
    # Decoder convention: -1=NO_SPIRO sentinel, 0..6 = positions
    # From A1's spiro_pos_class (0=NO_SPIRO, 1..7=positions 0..6):
    # Convert: (cls == 0) → -1, else (cls - 1)
    spiro_atom_positions = torch.where(
        spiro_pos_class == 0,
        torch.full_like(spiro_pos_class, -1),
        spiro_pos_class - 1,
    )

    # ── Decode scaffold (CPU/numpy) ───────────────────────────────
    # decode_v5_5_to_scaffold takes per-molecule numpy arrays.
    R_np   = R.cpu().numpy()
    F_np   = F.cpu().numpy()
    L_np   = L.cpu().numpy()
    B_size_np   = B_size.cpu().numpy()
    B_pos_np    = B_pos.cpu().numpy()
    B_parent_np = B_parent.cpu().numpy()
    B_bond_np   = B_bond.cpu().numpy()
    spiro_np    = spiro_atom_positions.cpu().numpy()

    bond_classes_list = []
    atom_mask_list = []
    arom_mask_list = []
    decode_failures = []  # indices that fail to decode

    # Placeholder atom_ids for the decoder (it needs an array of length
    # M_MAX but doesn't depend on actual values for bond construction).
    atom_ids_placeholder = np.zeros(M_MAX, dtype=np.int64)

    for i in range(B):
        try:
            bc, am = decode_v5_5_to_scaffold(
                R_np[i], F_np[i], L_np[i],
                B_size_np[i], B_pos_np[i], B_parent_np[i], B_bond_np[i],
                spiro_np[i], atom_ids_placeholder, M_MAX_out=M_MAX,
            )
            arm = aromatic_constraint_mask_v5_5(bc, am)
            bond_classes_list.append(bc)
            atom_mask_list.append(am)
            arom_mask_list.append(arm)
        except Exception as e:
            decode_failures.append((i, type(e).__name__, str(e)[:80]))
            # Insert empty scaffold so batch shapes stay aligned
            bond_classes_list.append(np.zeros((M_MAX, M_MAX), dtype=np.int64))
            atom_mask_list.append(np.zeros(M_MAX, dtype=bool))
            arom_mask_list.append(np.zeros(M_MAX, dtype=bool))

    bond_classes = torch.from_numpy(np.stack(bond_classes_list)).long().to(device)
    atom_mask    = torch.from_numpy(np.stack(atom_mask_list)).bool().to(device)
    arom_mask    = torch.from_numpy(np.stack(arom_mask_list)).bool().to(device)
    if debug and decode_failures:
        print(f'  Decode failures: {len(decode_failures)}/{B}')
        for idx, etype, emsg in decode_failures[:3]:
            print(f'    [{idx}] {etype}: {emsg}')

    # ── Stage 1C: A2 (atom IDs) ───────────────────────────────────
    atom_ids = A2.sample(
        bond_classes=bond_classes, atom_mask=atom_mask, arom_mask=arom_mask,
        condition=condition, n_steps=a2_steps,
        temperature=temperature, cfg_scale=cfg_scale, seed=seed + 2,
    )
    # Zero out atom_ids for decode-failure rows (they'd produce garbage)
    for i, _, _ in decode_failures:
        atom_ids[i] = 0

    if debug:
        print(f'  A2 done: atom_ids unique counts = '
              f'{torch.bincount(atom_ids.reshape(-1))[:6].cpu().tolist()}')

    # ── Stage 2: Terminal (functionalization) ─────────────────────
    fragment_ids = TERM.sample(
        scaffold_atom_ids=atom_ids,
        scaffold_bond_classes=bond_classes,
        scaffold_atom_mask=atom_mask,
        condition=condition, n_steps=term_steps,
        temperature=temperature, seed=seed + 3,
    )
    if debug:
        n_decorated = (fragment_ids > 0).sum(dim=1).float().mean()
        print(f'  Terminal done: mean decorations per molecule = {n_decorated:.2f}')

    # ── Stage 2 → assemble + sanitize ─────────────────────────────
    smiles_list = assemble_batch_to_smiles(
        atom_ids, bond_classes, atom_mask, fragment_ids,
    )
    return smiles_list

print('Pipeline function defined.')

Pipeline function defined.


## 5. Smoke test on 8 molecules (debug=True)

Tiny batch with diagnostic prints. Should take ~10 seconds. If this prints valid SMILES, the integration is healthy.

In [ ]:
import torch
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # suppress RDKit's noisy warnings

torch.manual_seed(SEED)
cond = torch.randn(8, 2, device=device)  # N(0,1) approximates train distribution

smiles = generate_batch(cond, debug=True)

print('\nGenerated SMILES:')
n_valid = 0
for i, smi in enumerate(smiles):
    if smi is None:
        print(f'  [{i}] None (assembly failed)')
    else:
        # Round-trip canonicalize
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            print(f'  [{i}] {smi}  (NOT parseable by RDKit)')
        else:
            canonical = Chem.MolToSmiles(mol)
            print(f'  [{i}] {canonical}')
            n_valid += 1
print(f'\nValid: {n_valid}/8 ({100*n_valid/8:.1f}%)')

In [ ]:
# Diagnostic: break down why assemble_molecule returns None
import numpy as np, torch
from meanflow.compose_full_molecule_zinc import _BOND_CLASS_TO_TYPE, _TERMINAL_SPECS
from rdkit import Chem

# Generate 64 samples and instrument each
torch.manual_seed(SEED + 9999)
cond = torch.randn(64, 2, device=device)

# Re-run pipeline up to assembly, capturing intermediate outputs
@torch.no_grad()
def generate_and_diagnose(cond):
    from meanflow.ring_layout_decoder import (
        decode_v5_5_to_scaffold, aromatic_constraint_mask_v5_5,
        M_MAX, R_MAX, P_MAX, B_LEN_MAX,
    )

    a1_out = A1.sample(condition=cond, n_steps=A1_STEPS,
                       cfg_scale=CFG_SCALE, seed=SEED+9999)
    R, F_, L = a1_out['R'], a1_out['F'], a1_out['L']
    spc = a1_out['spiro_pos_class']

    a3_spiro = torch.where(spc == 0, torch.full_like(spc, 7), spc - 1)
    a3_out = A3.sample(R=R, F_mat=F_, L_mat=L, spiro_pos=a3_spiro,
                       condition=cond, cfg_scale=CFG_SCALE, post_process=True,
                       seed=SEED+9999+1)

    spiro_dec = torch.where(spc == 0, torch.full_like(spc, -1), spc - 1)
    bc_list, am_list, arm_list = [], [], []
    decode_failures = []
    aid_pl = np.zeros(M_MAX, dtype=np.int64)
    for i in range(cond.shape[0]):
        try:
            bc, am = decode_v5_5_to_scaffold(
                R[i].cpu().numpy(), F_[i].cpu().numpy(), L[i].cpu().numpy(),
                a3_out['B_size'][i].cpu().numpy(), a3_out['B_pos'][i].cpu().numpy(),
                a3_out['B_parent'][i].cpu().numpy(), a3_out['B_bond'][i].cpu().numpy(),
                spiro_dec[i].cpu().numpy(), aid_pl, M_MAX_out=M_MAX,
            )
            arm = aromatic_constraint_mask_v5_5(bc, am)
            bc_list.append(bc); am_list.append(am); arm_list.append(arm)
        except Exception as e:
            decode_failures.append(i)
            bc_list.append(np.zeros((M_MAX, M_MAX), dtype=np.int64))
            am_list.append(np.zeros(M_MAX, dtype=bool))
            arm_list.append(np.zeros(M_MAX, dtype=bool))

    bond_classes = torch.from_numpy(np.stack(bc_list)).long().to(device)
    atom_mask    = torch.from_numpy(np.stack(am_list)).bool().to(device)
    arom_mask    = torch.from_numpy(np.stack(arm_list)).bool().to(device)

    atom_ids = A2.sample(bond_classes=bond_classes, atom_mask=atom_mask,
                         arom_mask=arom_mask, condition=cond,
                         n_steps=A2_STEPS, cfg_scale=CFG_SCALE, seed=SEED+9999+2)
    for i in decode_failures: atom_ids[i] = 0

    fragment_ids = TERM.sample(
        scaffold_atom_ids=atom_ids, scaffold_bond_classes=bond_classes,
        scaffold_atom_mask=atom_mask, condition=cond,
        n_steps=TERM_STEPS, seed=SEED+9999+3,
    )
    return atom_ids, bond_classes, atom_mask, fragment_ids, decode_failures

atom_ids, bond_classes, atom_mask, fragment_ids, decode_fails = generate_and_diagnose(cond)

# Now reimplement assemble_molecule's checks to bucket failures
N = atom_ids.shape[0]
failures = {
    'decode_failed': len(decode_fails),
    'atom_pad_at_active': 0,
    'invalid_bond_class': 0,
    'unknown_fragment_id': 0,
    'fragment_at_padded_atom': 0,
    'sanitize_failed': 0,
    'success': 0,
}
worst_fragment_ids = []
worst_atom_examples = []

for i in range(N):
    aid = atom_ids[i].cpu().numpy()
    bc  = bond_classes[i].cpu().numpy()
    am  = atom_mask[i].cpu().numpy()
    fid = fragment_ids[i].cpu().numpy()

    if i in decode_fails: continue

    # Check 1: atom_id=0 at active position
    pad_at_active = int(((aid == 0) & am).sum())
    if pad_at_active > 0:
        failures['atom_pad_at_active'] += 1
        if len(worst_atom_examples) < 3:
            worst_atom_examples.append((i, pad_at_active, int(am.sum())))
        continue

    # Check 2: bond_class outside [0,4]
    bad_bc = ((bc < 0) | (bc > 4)).any()
    if bad_bc:
        failures['invalid_bond_class'] += 1
        continue

    # Check 3: fragment_id out of vocab range OR at padded atom
    bad_fid = ((fid < 0) | (fid > 22)).any()
    if bad_fid:
        failures['unknown_fragment_id'] += 1
        if len(worst_fragment_ids) < 5:
            worst_fragment_ids.append((i, [int(f) for f in fid if f > 22 or f < 0]))
        continue

    frag_at_pad = int(((fid > 0) & ~am).sum())
    if frag_at_pad > 0:
        failures['fragment_at_padded_atom'] += 1
        continue

    # Check 4: try actually assembling
    from meanflow.compose_full_molecule_zinc import assemble_molecule
    mol = assemble_molecule(aid, bc, am, fid)
    if mol is None:
        failures['sanitize_failed'] += 1
    else:
        failures['success'] += 1

print(f'Failure breakdown across {N} samples:')
for k, v in failures.items():
    print(f'  {k:30s} {v:4d}  ({100*v/N:5.1f}%)')

if worst_atom_examples:
    print(f'\nExamples of atom_id=0 at active position:')
    for idx, n_bad, n_total in worst_atom_examples:
        print(f'  sample {idx}: {n_bad}/{n_total} active atoms have id=0')

if worst_fragment_ids:
    print(f'\nExamples of out-of-vocab fragment_ids:')
    for idx, bad_ids in worst_fragment_ids:
        print(f'  sample {idx}: bad ids = {bad_ids[:10]}')

In [ ]:
# What exactly is RDKit rejecting? Let assemble run without catching the exception.
from rdkit import Chem
from rdkit.Chem import RWMol, Atom
from meanflow.compose_full_molecule_zinc import (
    _vocab_id_to_atom, _BOND_CLASS_TO_TYPE, _TERMINAL_SPECS,
)

# Find all sanitize-failed samples and diagnose
sanitize_errors = {}
example_smiles_pre_sanitize = []

for i in range(64):
    if i in decode_fails: continue
    aid = atom_ids[i].cpu().numpy()
    bc  = bond_classes[i].cpu().numpy()
    am  = atom_mask[i].cpu().numpy()
    fid = fragment_ids[i].cpu().numpy()

    # Reimplement assemble WITHOUT sanitize to get the SMILES that would have been produced
    try:
        mol = Chem.RWMol()
        slot_to_rdkit = {}
        for j in range(len(aid)):
            if not am[j]: continue
            vid = int(aid[j])
            if vid == 0: continue
            element, is_arom = _vocab_id_to_atom(vid)
            atom = Atom(element)
            atom.SetIsAromatic(is_arom)
            slot_to_rdkit[j] = mol.AddAtom(atom)
        for j in range(len(aid)):
            if j not in slot_to_rdkit: continue
            for k in range(j + 1, len(aid)):
                if k not in slot_to_rdkit: continue
                bcj = int(bc[j, k])
                if bcj == 0: continue
                mol.AddBond(slot_to_rdkit[j], slot_to_rdkit[k], _BOND_CLASS_TO_TYPE[bcj])
        # Grafts
        for j in range(len(fid)):
            if not am[j]: continue
            f = int(fid[j])
            if f == 0: continue
            spec = _TERMINAL_SPECS.get(f)
            if spec is None: continue
            frag_idx_to_rdkit = {}
            for k, (element, is_arom, num_h) in enumerate(spec['atoms']):
                a = Atom(element); a.SetIsAromatic(is_arom)
                if num_h > 0: a.SetNumExplicitHs(num_h)
                frag_idx_to_rdkit[k] = mol.AddAtom(a)
            mol.AddBond(slot_to_rdkit[j], frag_idx_to_rdkit[0], spec['attach'])
            for (a, b, bt) in spec['bonds']:
                mol.AddBond(frag_idx_to_rdkit[a], frag_idx_to_rdkit[b], bt)

        # Now try sanitize and capture the error
        try:
            Chem.SanitizeMol(mol)
            # If we got here, sample is fine
        except Exception as e:
            err_class = type(e).__name__
            err_msg = str(e)[:100]
            key = err_msg.split(':')[0][:50]  # bucket by error type
            sanitize_errors[key] = sanitize_errors.get(key, 0) + 1
            if len(example_smiles_pre_sanitize) < 5:
                # Get SMILES before sanitize
                try:
                    smi_no_san = Chem.MolToSmiles(mol, kekuleSmiles=False)
                    example_smiles_pre_sanitize.append((i, key, smi_no_san[:120]))
                except:
                    example_smiles_pre_sanitize.append((i, key, '<unprintable>'))
    except Exception as e:
        print(f'sample {i}: pre-sanitize crash {type(e).__name__}: {e}')

print(f'\nSanitize error buckets:')
for err, count in sorted(sanitize_errors.items(), key=lambda kv: -kv[1]):
    print(f'  {count:3d}× {err}')

print(f'\nExample pre-sanitize SMILES (what was rejected):')
for idx, err, smi in example_smiles_pre_sanitize:
    print(f'  [{idx}] ({err[:30]})  {smi}')

## 6. Generate N_SAMPLES molecules at varied conditions

Now scale up. Conditions are drawn from N(0,1) per axis — this approximates the Z-scored training distribution (`logP_norm`, `SAS_norm` both have mean=0, std=1 by construction).

Wall-clock: about **30-60s per 64 molecules** depending on GPU. 1024 ≈ 8-15 min.

In [ ]:
import torch, time
from tqdm import tqdm

all_smiles = []
all_conds = []

torch.manual_seed(SEED + 100)
n_batches = (N_SAMPLES + BATCH_SIZE - 1) // BATCH_SIZE
t_start = time.time()

for batch_idx in tqdm(range(n_batches), desc='generating'):
    bs = min(BATCH_SIZE, N_SAMPLES - len(all_smiles))
    cond = torch.randn(bs, 2, device=device)
    batch_smiles = generate_batch(cond, seed=SEED + batch_idx * 7)
    all_smiles.extend(batch_smiles)
    all_conds.append(cond.cpu())
    if len(all_smiles) >= N_SAMPLES:
        break

elapsed = time.time() - t_start
all_conds = torch.cat(all_conds, dim=0)[:N_SAMPLES]
all_smiles = all_smiles[:N_SAMPLES]
print(f'\nGenerated {len(all_smiles)} samples in {elapsed:.0f}s ({elapsed/len(all_smiles)*1000:.0f}ms/molecule)')

## 7. Compute V·U·N

Validity = fraction of samples that RDKit can sanitize into a valid molecule.  
Uniqueness = fraction of unique canonical SMILES among valid samples.  
Novelty = fraction of valid samples NOT in training set.

In [ ]:
import pickle
from rdkit import Chem

# Build training set of canonical SMILES from the labels.pkl
print('Loading training set for novelty check...')
with open(LABELS_PKL, 'rb') as f:
    labels = pickle.load(f)
if not isinstance(labels, list):
    labels = labels.get('labels', labels)

train_canonical = set()
for lab in labels:
    smi = lab.get('smi', None)
    if smi is None: continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None: continue
    train_canonical.add(Chem.MolToSmiles(mol))
print(f'Training canonical SMILES: {len(train_canonical):,}')

# ── Validity ─────────────────────────────────────────────────────
valid_canonical = []
n_assembly_fail = 0
n_parse_fail = 0
for smi in all_smiles:
    if smi is None:
        n_assembly_fail += 1
        continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        n_parse_fail += 1
        continue
    valid_canonical.append(Chem.MolToSmiles(mol))

n_total = len(all_smiles)
n_valid = len(valid_canonical)
validity = n_valid / n_total

# ── Uniqueness ───────────────────────────────────────────────────
unique_valid = set(valid_canonical)
uniqueness = len(unique_valid) / max(n_valid, 1)

# ── Novelty ──────────────────────────────────────────────────────
novel = unique_valid - train_canonical
novelty = len(novel) / max(len(unique_valid), 1)

# ── Report ───────────────────────────────────────────────────────
print('\n' + '=' * 50)
print(f' V·U·N evaluation on {n_total} generated samples')
print('=' * 50)
print(f' Validity:     {validity*100:5.1f}%   ({n_valid}/{n_total})')
print(f' Uniqueness:   {uniqueness*100:5.1f}%   ({len(unique_valid)}/{n_valid} valid)')
print(f' Novelty:      {novelty*100:5.1f}%   ({len(novel)}/{len(unique_valid)} unique)')
print()
print(f' Failure breakdown:')
print(f'   Assembly failed (None):     {n_assembly_fail}')
print(f'   RDKit parse failed:         {n_parse_fail}')
print(f'   Total invalid:              {n_assembly_fail + n_parse_fail}')
print()
print(f' Sample valid molecules:')
for smi in list(unique_valid)[:10]:
    print(f'   {smi}')

In [ ]:
import sys, inspect
# Force fresh import
for k in list(sys.modules):
    if 'compose_full_molecule_zinc' in k or k.startswith('meanflow'):
        del sys.modules[k]

from meanflow import compose_full_molecule_zinc as cfm
print("Module path:", cfm.__file__)
print("Has _vocab_id_to_charge?", hasattr(cfm, "_vocab_id_to_charge"))
print("Has _max_valence_for?", hasattr(cfm, "_max_valence_for"))
src = inspect.getsource(cfm.assemble_molecule)
print("assemble_molecule contains SetFormalCharge?:", "SetFormalCharge" in src)
print("assemble_molecule contains 'graft validator' check?:",
      "current_val + attach_order > max_val" in src)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 8b. Failure dumper — diagnose the 169 (assembly-failed) molecules
# ═══════════════════════════════════════════════════════════════════════
#
# What this cell does:
#   Re-runs the generation pipeline for the EXACT same 1024 samples whose
#   V·U·N was just computed (same seeds, same conditions), but this time
#   we capture all intermediate per-molecule tensors. For each None-SMILES
#   sample we record:
#     - the assembly-pre-sanitize SMILES (or "<unprintable>")
#     - the exact RDKit error class & message
#     - failure category (decode/grafting/sanitize)
#     - sanitize sub-bucket (kekulize / valence / element / etc.)
#     - scaffold metadata: n_active_atoms, n_aromatic_atoms, n_rings,
#       atom-id histogram, terminal-id histogram (which spec was grafted)
#     - the conditioning vector used
#
# Output:
#   - In-notebook grouped table by error bucket (top 5 examples each)
#   - JSON file at $BASE/failure_dump_v5_5_1.json for offline analysis
#
# What we're hoping to learn from the 169 failures:
#   Path A — Dominated by one sub-bucket (e.g., 80% are "Can't kekulize"
#            with 5-ring all-c scaffolds): single targeted patch.
#   Path B — Long tail with no single dominant pattern: validity ceiling
#            from this model is ~85% without retraining.
#   Path C — Cluster of failures involving a specific terminal class
#            (e.g., class 22 isonitrile): one-line spec fix.
# ═══════════════════════════════════════════════════════════════════════

import numpy as np
import torch
import json
import os
from collections import Counter, defaultdict
from rdkit import Chem
from rdkit.Chem import RWMol, Atom

# Reuse pipeline internals
from meanflow.compose_full_molecule_zinc import (
    _vocab_id_to_atom, _vocab_id_to_charge, _BOND_CLASS_TO_TYPE,
    _TERMINAL_SPECS, ATOM_VOCAB, assemble_molecule,
)
from meanflow.ring_layout_decoder import (
    decode_v5_5_to_scaffold, aromatic_constraint_mask_v5_5,
    M_MAX, R_MAX, P_MAX, B_LEN_MAX,
)

# Sanity check: we need all_smiles and all_conds from the V·U·N cell.
assert len(all_smiles) == N_SAMPLES, (
    f"all_smiles has {len(all_smiles)} entries, expected {N_SAMPLES}. "
    f"Run the generation cell + V·U·N cell first."
)
assert all_conds.shape == (N_SAMPLES, 2), (
    f"all_conds shape {tuple(all_conds.shape)}, expected ({N_SAMPLES}, 2)"
)

# ─── Identify the failure indices ─────────────────────────────────────
failure_indices = [i for i, smi in enumerate(all_smiles) if smi is None]
n_failures = len(failure_indices)
print(f"Total failures to diagnose: {n_failures}/{N_SAMPLES} "
      f"({100*n_failures/N_SAMPLES:.1f}%)")

if n_failures == 0:
    print("No failures — nothing to dump.")
else:
    # ─── Re-run generation pipeline to recover intermediate tensors ────
    # We need atom_ids / bond_classes / fragment_ids per failed sample.
    # The generation cell only kept the SMILES; we have to regenerate.
    # Seeds match the V·U·N run exactly: SEED + batch_idx * 7 per batch.
    #
    # Note: this re-runs the FULL 1024, not just the failures — we can't
    # selectively run individual samples through the diffusion pipeline
    # because batches share random state. Re-runs cleanly because the
    # pipeline is deterministic given the seed.
    print(f"\nRe-running pipeline to recover intermediate tensors...")

    all_atom_ids       = []  # list of (M_MAX,) numpy arrays
    all_bond_classes   = []  # list of (M_MAX, M_MAX) numpy arrays
    all_atom_mask      = []  # list of (M_MAX,) bool numpy arrays
    all_fragment_ids   = []  # list of (M_MAX,) numpy arrays
    decode_failure_idx = set()  # global indices where decode_v5_5 raised

    @torch.no_grad()
    def regenerate_and_capture(cond_batch, seed):
        """Re-run pipeline for one batch, return (atom_ids, bond_classes,
        atom_mask, fragment_ids, decode_failed_indices_in_batch)."""
        B = cond_batch.shape[0]
        # A1
        a1_out = A1.sample(condition=cond_batch, n_steps=A1_STEPS,
                           temperature=TEMPERATURE, cfg_scale=CFG_SCALE, seed=seed)
        R, F_, L = a1_out['R'], a1_out['F'], a1_out['L']
        spc = a1_out['spiro_pos_class']
        # A3 spiro convention conversion
        a3_spiro = torch.where(spc == 0, torch.full_like(spc, 7), spc - 1)
        a3_out = A3.sample(R=R, F_mat=F_, L_mat=L, spiro_pos=a3_spiro,
                           condition=cond_batch, cfg_scale=CFG_SCALE,
                           temperature=TEMPERATURE, post_process=True,
                           seed=seed + 1)
        # Decoder spiro convention
        spiro_dec = torch.where(spc == 0, torch.full_like(spc, -1), spc - 1)

        bc_list, am_list, arm_list = [], [], []
        local_decode_fail = []
        aid_pl = np.zeros(M_MAX, dtype=np.int64)
        for i in range(B):
            try:
                bc, am = decode_v5_5_to_scaffold(
                    R[i].cpu().numpy(), F_[i].cpu().numpy(), L[i].cpu().numpy(),
                    a3_out['B_size'][i].cpu().numpy(),
                    a3_out['B_pos'][i].cpu().numpy(),
                    a3_out['B_parent'][i].cpu().numpy(),
                    a3_out['B_bond'][i].cpu().numpy(),
                    spiro_dec[i].cpu().numpy(), aid_pl, M_MAX_out=M_MAX,
                )
                arm = aromatic_constraint_mask_v5_5(bc, am)
                bc_list.append(bc); am_list.append(am); arm_list.append(arm)
            except Exception:
                local_decode_fail.append(i)
                bc_list.append(np.zeros((M_MAX, M_MAX), dtype=np.int64))
                am_list.append(np.zeros(M_MAX, dtype=bool))
                arm_list.append(np.zeros(M_MAX, dtype=bool))

        bond_classes = torch.from_numpy(np.stack(bc_list)).long().to(device)
        atom_mask    = torch.from_numpy(np.stack(am_list)).bool().to(device)
        arom_mask    = torch.from_numpy(np.stack(arm_list)).bool().to(device)

        atom_ids = A2.sample(bond_classes=bond_classes, atom_mask=atom_mask,
                             arom_mask=arom_mask, condition=cond_batch,
                             n_steps=A2_STEPS, temperature=TEMPERATURE,
                             cfg_scale=CFG_SCALE, seed=seed + 2)
        for i in local_decode_fail: atom_ids[i] = 0

        fragment_ids = TERM.sample(
            scaffold_atom_ids=atom_ids, scaffold_bond_classes=bond_classes,
            scaffold_atom_mask=atom_mask, condition=cond_batch,
            n_steps=TERM_STEPS, temperature=TEMPERATURE, seed=seed + 3,
        )
        return (atom_ids.cpu().numpy(), bond_classes.cpu().numpy(),
                atom_mask.cpu().numpy(), fragment_ids.cpu().numpy(),
                local_decode_fail)

    # Match the exact batching/seeding of the V·U·N generation cell:
    #   torch.manual_seed(SEED + 100); per batch: cond = randn(bs, 2);
    #   batch_smiles = generate_batch(cond, seed=SEED + batch_idx * 7)
    torch.manual_seed(SEED + 100)
    n_batches = (N_SAMPLES + BATCH_SIZE - 1) // BATCH_SIZE
    cursor = 0
    for batch_idx in range(n_batches):
        bs = min(BATCH_SIZE, N_SAMPLES - cursor)
        if bs <= 0: break
        cond_batch = torch.randn(bs, 2, device=device)
        a, b, m, f, ldf = regenerate_and_capture(cond_batch, seed=SEED + batch_idx * 7)
        for i in range(bs):
            all_atom_ids.append(a[i])
            all_bond_classes.append(b[i])
            all_atom_mask.append(m[i])
            all_fragment_ids.append(f[i])
            if i in ldf:
                decode_failure_idx.add(cursor + i)
        cursor += bs
    print(f"  Recovered tensors for {len(all_atom_ids)} samples; "
          f"{len(decode_failure_idx)} decode failures.")

    # ─── Sanity check: do the recovered SMILES match all_smiles? ───────
    # If the pipeline is deterministic given seed, this should match
    # exactly. A mismatch would indicate the failure dump is contaminated.
    mismatch = 0
    for i in range(N_SAMPLES):
        recovered = assemble_molecule(
            all_atom_ids[i], all_bond_classes[i],
            all_atom_mask[i], all_fragment_ids[i],
        )
        recovered_smi = Chem.MolToSmiles(recovered) if recovered else None
        # Compare against canonicalized all_smiles
        orig = all_smiles[i]
        if orig is None and recovered_smi is None:
            continue
        if orig is None or recovered_smi is None:
            mismatch += 1
            continue
        orig_canon = Chem.MolToSmiles(Chem.MolFromSmiles(orig)) if orig else None
        if orig_canon != recovered_smi:
            mismatch += 1
    if mismatch == 0:
        print(f"  ✓ All {N_SAMPLES} samples reproduced identically.")
    else:
        print(f"  ⚠ {mismatch} samples differ between V·U·N run and "
              f"failure-dump regeneration. Diagnoses below may be slightly "
              f"off if your run lands on a different RNG state. The error-"
              f"bucket distribution should still be representative.")

    # ─── Diagnose each failure ──────────────────────────────────────────
    print(f"\nDiagnosing {n_failures} failures...")
    failure_records = []

    for i in failure_indices:
        aid = all_atom_ids[i]
        bc  = all_bond_classes[i]
        am  = all_atom_mask[i]
        fid = all_fragment_ids[i]
        cond_i = all_conds[i].tolist()

        # Metadata
        n_active = int(am.sum())
        n_aromatic = int(((bc == 2).any(axis=1)).sum())  # rough: atoms with any aromatic bond
        n_bonds_total = int((bc > 0).sum() // 2)  # symmetric
        atom_hist = Counter(int(a) for j, a in enumerate(aid) if am[j])
        atom_hist_sym = {ATOM_VOCAB[k] if k < len(ATOM_VOCAB) else f"vid{k}": v
                         for k, v in atom_hist.items()}
        frag_hist = Counter(int(f) for j, f in enumerate(fid) if am[j] and f > 0)

        # Category 1: decode failure
        if i in decode_failure_idx:
            failure_records.append({
                'idx': i, 'category': 'decode_failed',
                'error_class': None, 'error_msg': 'decode_v5_5_to_scaffold raised',
                'pre_sanitize_smiles': None,
                'n_active_atoms': n_active, 'n_bonds_total': n_bonds_total,
                'atom_histogram': atom_hist_sym, 'fragment_histogram': dict(frag_hist),
                'cond_logP_norm': cond_i[0], 'cond_SAS_norm': cond_i[1],
            })
            continue

        # Try to build the molecule WITHOUT sanitize and capture what happens
        try:
            mol = RWMol()
            slot_to_rdkit = {}
            for j in range(len(aid)):
                if not am[j]: continue
                vid = int(aid[j])
                if vid == 0: continue
                elem, is_arom = _vocab_id_to_atom(vid)
                chg = _vocab_id_to_charge(vid)
                a = Atom(elem); a.SetIsAromatic(is_arom)
                if chg != 0: a.SetFormalCharge(chg)
                slot_to_rdkit[j] = mol.AddAtom(a)
            # Scaffold bonds
            for j in range(len(aid)):
                if j not in slot_to_rdkit: continue
                for k in range(j + 1, len(aid)):
                    if k not in slot_to_rdkit: continue
                    b_ = int(bc[j, k])
                    if b_ == 0: continue
                    mol.AddBond(slot_to_rdkit[j], slot_to_rdkit[k],
                                _BOND_CLASS_TO_TYPE[b_])
            # Terminal grafts (mirror the v5.5.1 valence-aware logic
            # so the dump reflects what assemble_molecule actually did)
            from meanflow.compose_full_molecule_zinc import (
                _current_explicit_valence, _max_valence_for, _BOND_ORDER,
            )
            for j in range(len(fid)):
                if not am[j]: continue
                f_ = int(fid[j])
                if f_ == 0: continue
                spec = _TERMINAL_SPECS.get(f_)
                if spec is None: continue
                host_idx = slot_to_rdkit[j]
                host = mol.GetAtomWithIdx(host_idx)
                attach_order = _BOND_ORDER.get(spec['attach'], 1.0)
                cur_val = _current_explicit_valence(mol, host_idx)
                max_val = _max_valence_for(host.GetSymbol(), host.GetFormalCharge())
                if cur_val + attach_order > max_val: continue
                if host.GetIsAromatic() and host.GetDegree() >= 3: continue
                frag_map = {}
                for k, (elem, is_arom, num_h) in enumerate(spec['atoms']):
                    a = Atom(elem); a.SetIsAromatic(is_arom)
                    if num_h > 0: a.SetNumExplicitHs(num_h)
                    frag_map[k] = mol.AddAtom(a)
                mol.AddBond(host_idx, frag_map[0], spec['attach'])
                for (a, b_, bt) in spec['bonds']:
                    mol.AddBond(frag_map[a], frag_map[b_], bt)

            # Try sanitize
            try:
                Chem.SanitizeMol(mol)
                # If we get here, the molecule is actually fine —
                # something else went wrong (shouldn't happen since this
                # mirrors assemble_molecule, but flag it).
                failure_records.append({
                    'idx': i, 'category': 'mismatch_sanitize_ok',
                    'error_class': None, 'error_msg': 'sanitize passed in dumper but failed in pipeline',
                    'pre_sanitize_smiles': Chem.MolToSmiles(mol, kekuleSmiles=False),
                    'n_active_atoms': n_active, 'n_bonds_total': n_bonds_total,
                    'atom_histogram': atom_hist_sym, 'fragment_histogram': dict(frag_hist),
                    'cond_logP_norm': cond_i[0], 'cond_SAS_norm': cond_i[1],
                })
            except Exception as san_e:
                err_class = type(san_e).__name__
                err_msg = str(san_e)
                # Bucket: pull the leading phrase (RDKit format is consistent)
                first_line = err_msg.split('\n')[0]
                bucket = first_line.split(':')[0][:60]
                if 'kekulize' in first_line.lower():
                    bucket = 'kekulize'
                elif 'explicit valence' in first_line.lower():
                    # Extract element from "atom # N X, V, is greater"
                    import re
                    m = re.search(r'atom # \d+ (\w+), (\d+),', first_line)
                    if m:
                        bucket = f'valence_{m.group(1)}_{m.group(2)}'
                    else:
                        bucket = 'valence_unknown'
                # Get pre-sanitize SMILES for inspection
                try:
                    pre_smi = Chem.MolToSmiles(mol, kekuleSmiles=False, canonical=False)
                except Exception:
                    pre_smi = '<unprintable>'
                failure_records.append({
                    'idx': i, 'category': 'sanitize_failed',
                    'sanitize_bucket': bucket,
                    'error_class': err_class, 'error_msg': first_line[:160],
                    'pre_sanitize_smiles': pre_smi[:200],
                    'n_active_atoms': n_active, 'n_bonds_total': n_bonds_total,
                    'atom_histogram': atom_hist_sym, 'fragment_histogram': dict(frag_hist),
                    'cond_logP_norm': cond_i[0], 'cond_SAS_norm': cond_i[1],
                })
        except Exception as build_e:
            # Pre-sanitize crash (shouldn't happen with v5.5.1 but possible)
            failure_records.append({
                'idx': i, 'category': 'pre_sanitize_crash',
                'error_class': type(build_e).__name__,
                'error_msg': str(build_e).split('\n')[0][:160],
                'pre_sanitize_smiles': None,
                'n_active_atoms': n_active, 'n_bonds_total': n_bonds_total,
                'atom_histogram': atom_hist_sym, 'fragment_histogram': dict(frag_hist),
                'cond_logP_norm': cond_i[0], 'cond_SAS_norm': cond_i[1],
            })

    # ─── Summary report ────────────────────────────────────────────────
    print(f"\n{'='*68}")
    print(f"FAILURE DUMP — {len(failure_records)} failures categorized")
    print(f"{'='*68}\n")

    # By category
    cat_counts = Counter(r['category'] for r in failure_records)
    print("Category breakdown:")
    for cat, count in cat_counts.most_common():
        print(f"  {cat:30s}  {count:4d}  ({100*count/n_failures:5.1f}%)")

    # By sanitize sub-bucket
    bucket_counts = Counter(
        r.get('sanitize_bucket', 'n/a') for r in failure_records
        if r['category'] == 'sanitize_failed'
    )
    if bucket_counts:
        print("\nSanitize sub-buckets:")
        for buck, count in bucket_counts.most_common():
            print(f"  {buck:30s}  {count:4d}")

    # Terminal class involvement: which terminal ids show up disproportionately
    # in failures? Compare frequency in failures vs frequency in all 1024.
    print("\nTerminal class involvement in failures (top 10):")
    fail_term = Counter()
    all_term  = Counter()
    for i in range(N_SAMPLES):
        for f_ in all_fragment_ids[i]:
            if f_ > 0: all_term[int(f_)] += 1
    for r in failure_records:
        for k, v in r['fragment_histogram'].items():
            fail_term[k] += v
    total_fail = sum(fail_term.values()) or 1
    total_all  = sum(all_term.values()) or 1
    print(f"  {'class':<6} {'in_fails':>10} {'in_all':>10} {'rel_freq':>10}")
    for cls, fc in fail_term.most_common(10):
        ac = all_term.get(cls, 0)
        fail_pct = fc / total_fail * 100
        all_pct  = ac / total_all * 100
        rel      = fail_pct / all_pct if all_pct > 0 else float('inf')
        marker = '  ⚠' if rel > 1.5 else ''
        print(f"  {cls:<6d} {fc:>10d} {ac:>10d} {rel:>9.2f}x{marker}")

    # Top 5 examples per sub-bucket
    print(f"\n{'='*68}")
    print("TOP 5 EXAMPLES PER SUB-BUCKET (pre-sanitize SMILES)")
    print(f"{'='*68}\n")
    by_bucket = defaultdict(list)
    for r in failure_records:
        key = r.get('sanitize_bucket', r['category'])
        by_bucket[key].append(r)

    for bucket, recs in sorted(by_bucket.items(), key=lambda kv: -len(kv[1])):
        print(f"\n── {bucket}  (n={len(recs)}) ──")
        for r in recs[:5]:
            smi = r.get('pre_sanitize_smiles') or '(no smiles)'
            errsnip = (r.get('error_msg') or '')[:70]
            print(f"  [{r['idx']:4d}] n_atoms={r['n_active_atoms']:2d}  "
                  f"frags={dict(list(r['fragment_histogram'].items())[:4])}")
            print(f"         {smi[:110]}")
            if errsnip and bucket != r.get('sanitize_bucket'):
                print(f"         err: {errsnip}")

    # ─── Save full JSON dump ───────────────────────────────────────────
    dump_path = f'{BASE}/failure_dump_v5_5_1.json'
    with open(dump_path, 'w') as f:
        json.dump({
            'n_samples_total': N_SAMPLES,
            'n_failures': n_failures,
            'failures': failure_records,
            'category_counts': dict(cat_counts),
            'sanitize_bucket_counts': dict(bucket_counts),
            'terminal_class_freq_in_failures': dict(fail_term),
            'terminal_class_freq_overall':    dict(all_term),
        }, f, indent=2, default=str)
    print(f"\n✓ Full dump saved to: {dump_path}")
    print(f"  ({n_failures} records, {os.path.getsize(dump_path)/1024:.0f} KB)")

## 8. Controllability — Pearson r for logP and SAS

If A1's CFG conditioning is working end-to-end, the generated molecules' actual logP/SAS (computed via RDKit) should correlate with the target conditions we sampled.

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Crippen
from rdkit.Chem import RDConfig
import sys, os

# RDKit's SAS calculator is a contrib script
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
try:
    import sascorer
    HAS_SAS = True
except ImportError:
    HAS_SAS = False
    print('WARNING: sascorer not available; skipping SAS correlation.')

# Need the original (un-normalized) mean/std for conversion
import pandas as pd
df = pd.read_csv(f'{BASE}/dataset_augmented.csv')
logP_mean = df['logP'].mean(); logP_std = df['logP'].std()
SAS_mean  = df['SAS'].mean();  SAS_std  = df['SAS'].std()
print(f'Train logP: mean={logP_mean:.3f}, std={logP_std:.3f}')
print(f'Train SAS:  mean={SAS_mean:.3f},  std={SAS_std:.3f}')

# Compute properties for each valid generated molecule
target_logP_norm, actual_logP_norm = [], []
target_SAS_norm,  actual_SAS_norm  = [], []

conds_np = all_conds.numpy()  # (N, 2): [:, 0] = logP_norm, [:, 1] = SAS_norm

for i, smi in enumerate(all_smiles):
    if smi is None: continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None: continue

    # logP: compute first, then append both together — atomic
    try:
        logP_val = Crippen.MolLogP(mol)
        logP_norm_val = (logP_val - logP_mean) / logP_std
    except Exception:
        continue  # skip mol entirely if even logP fails
    target_logP_norm.append(conds_np[i, 0])
    actual_logP_norm.append(logP_norm_val)

    # SAS: independent try/except, atomic append
    # (a SAS failure must not corrupt the logP collection above
    #  nor the target/actual SAS list alignment)
    if HAS_SAS:
        try:
            sas_val = sascorer.calculateScore(mol)
            sas_norm_val = (sas_val - SAS_mean) / SAS_std
            if not np.isfinite(sas_norm_val):
                raise ValueError('non-finite SAS')
        except Exception:
            continue
        target_SAS_norm.append(conds_np[i, 1])
        actual_SAS_norm.append(sas_norm_val)

target_logP_norm = np.array(target_logP_norm)
actual_logP_norm = np.array(actual_logP_norm)
logP_r = np.corrcoef(target_logP_norm, actual_logP_norm)[0, 1]
print(f'\nControllability (Pearson r on Z-scored values):')
print(f'  logP: r = {logP_r:.3f}   (N={len(target_logP_norm)})')

if HAS_SAS and target_SAS_norm:
    target_SAS_norm = np.array(target_SAS_norm)
    actual_SAS_norm = np.array(actual_SAS_norm)
    # Defensive: confirm sync (shouldn't fire, but catches future regressions)
    assert len(target_SAS_norm) == len(actual_SAS_norm), \
        f'SAS list desync: {len(target_SAS_norm)} vs {len(actual_SAS_norm)}'
    SAS_r = np.corrcoef(target_SAS_norm, actual_SAS_norm)[0, 1]
    print(f'  SAS:  r = {SAS_r:.3f}   (N={len(target_SAS_norm)})')

# ── Plot ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
n_plots = 2 if (HAS_SAS and len(target_SAS_norm) > 0) else 1
fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 5), squeeze=False)
axes = axes[0]

axes[0].scatter(target_logP_norm, actual_logP_norm, alpha=0.3, s=10)
lim = max(abs(target_logP_norm).max(), abs(actual_logP_norm).max(), 3)
axes[0].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='y=x')
axes[0].set_xlabel('Target logP (Z-scored)')
axes[0].set_ylabel('Actual logP (Z-scored)')
axes[0].set_title(f'logP controllability: r = {logP_r:.3f}')
axes[0].grid(alpha=0.3); axes[0].legend()
axes[0].set_xlim(-lim, lim); axes[0].set_ylim(-lim, lim)

if HAS_SAS and len(target_SAS_norm) > 0:
    axes[1].scatter(target_SAS_norm, actual_SAS_norm, alpha=0.3, s=10)
    lim = max(abs(target_SAS_norm).max(), abs(actual_SAS_norm).max(), 3)
    axes[1].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='y=x')
    axes[1].set_xlabel('Target SAS (Z-scored)')
    axes[1].set_ylabel('Actual SAS (Z-scored)')
    axes[1].set_title(f'SAS controllability: r = {SAS_r:.3f}')
    axes[1].grid(alpha=0.3); axes[1].legend()
    axes[1].set_xlim(-lim, lim); axes[1].set_ylim(-lim, lim)

plt.tight_layout(); plt.show()

## 9. Sample gallery — visualize top-20 generated molecules

Quick eyeball check of chemistry quality.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem

# Take 20 unique valid molecules, prefer larger ones (more interesting)
unique_list = sorted(unique_valid,
                     key=lambda s: -Chem.MolFromSmiles(s).GetNumHeavyAtoms())
gallery_smiles = unique_list[:20]
gallery_mols = [Chem.MolFromSmiles(s) for s in gallery_smiles]
for m in gallery_mols:
    if m: AllChem.Compute2DCoords(m)

img = Draw.MolsToGridImage(
    gallery_mols, molsPerRow=4, subImgSize=(280, 220),
    legends=[f'{i}: {Chem.MolFromSmiles(s).GetNumHeavyAtoms()}HA'
             for i, s in enumerate(gallery_smiles)],
    useSVG=False,
)
img

## Done

**Interpretation guide for V·U·N + controllability results:**

| Metric | Healthy | Acceptable | Problematic |
|---|---|---|---|
| Validity | ≥ 80% | 50-80% | < 50% |
| Uniqueness | ≥ 95% | 90-95% | < 90% (mode collapse) |
| Novelty | ≥ 95% | 90-95% | < 90% (memorization) |
| logP r | ≥ 0.6 | 0.3-0.6 | < 0.3 |
| SAS r | ≥ 0.4 | 0.2-0.4 | < 0.2 |

**If validity is low**, check the assembly-vs-parse breakdown — assembly failures are bond-class issues (likely A1+A3 producing weird bond patterns); parse failures are sanitization issues (likely A2 producing wrong-valence atoms).

**If uniqueness is low** (mode collapse), tune CFG: try cfg_scale=2.0 to push generation off the prior.

**If controllability is weak** but validity is high, A1's conditioning is being washed out at high CFG. Try cfg_scale=1.0.

**Next steps if metrics are below targets:** revisit Terminal training (class_weight_smoothing=5.0), or add ring-composition conditioning to A3 (the upgrade path noted in the design doc).